# 05 - Download Data dari TradingView (Premium)

> ⚠️ **Jalankan notebook ini di mesin LOKAL, bukan di Kaggle.**  
> Kaggle memblokir koneksi WebSocket ke TradingView.

**Tujuan:** Unduh 6000 bar 15-menit untuk SPX, NIFTY, FTSE → simpan ke  
`data/raw/{symbol}_15m.parquet` sesuai format yang diharapkan `src/data.py`.

**Alur:**
1. Install `tvdatafeed` dari GitHub
2. Login dengan akun TradingView premium (via `getpass` — tidak tersimpan di disk)
3. Probe kombinasi symbol/exchange yang valid di TradingView
4. Download 6000 bar per indeks dengan retry
5. Validasi hasil (jumlah bar, null, duplikat, rentang tanggal)
6. Simpan ke `data/raw/`
7. Instruksi upload ke Kaggle sebagai Dataset


## 1. Setup

In [ ]:
import sys, subprocess
from pathlib import Path

# Pastikan tvdatafeed terinstal
try:
    from tvDatafeed import TvDatafeed, Interval  # noqa: F401
    print('tvDatafeed sudah terinstal.')
except ImportError:
    print('Menginstal tvdatafeed dari GitHub ...')
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q',
         'git+https://github.com/rongardF/tvdatafeed.git'],
        check=True,
    )
    from tvDatafeed import TvDatafeed, Interval
    print('tvDatafeed berhasil diinstal.')

# Temukan repo root
REPO = Path().resolve()
while not (REPO / 'configs' / 'config.yaml').exists() and REPO != REPO.parent:
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import yaml
import pandas as pd

cfg        = yaml.safe_load((REPO / 'configs' / 'config.yaml').read_text())
N_BARS     = cfg['data']['n_bars']      # 6000
N_BARS_MIN = cfg['data']['n_bars_min']  # 3000
INDICES    = cfg['data']['indices']

from src.paths import Paths
paths = Paths()

print(f'Repo      : {REPO}')
print(f'Raw dir   : {paths.raw}')
print(f'N_BARS    : {N_BARS}')
print(f'Indices   : {[i["symbol"] for i in INDICES]}')


## 2. Login TradingView Premium

Password diketik via `getpass` — **tidak tampil di layar dan tidak tersimpan di disk/notebook.**


In [ ]:
from getpass import getpass
from tvDatafeed import TvDatafeed, Interval

tv_username = input('TradingView username / email : ')
tv_password = getpass('TradingView password        : ')

tv = TvDatafeed(tv_username, tv_password)
print('Login selesai. Objek TvDatafeed siap.')


## 3. Probe Symbol / Exchange yang Valid

TradingView kadang menggunakan nama exchange berbeda dari bursa sesungguhnya.  
Cell ini menguji beberapa kandidat dengan `n_bars=5` dan melaporkan yang berhasil.


In [ ]:
CANDIDATES = {
    'SPX': [
        ('SPX',      'SP',          'S&P 500 di Standard & Poors'),
        ('SPX',      'NYSE',        'S&P 500 di NYSE (config.yaml saat ini)'),
        ('SP500USD', 'OANDA',       'S&P 500 CFD di OANDA'),
        ('US500',    'TVC',         'S&P 500 di TVC'),
    ],
    'NIFTY': [
        ('NIFTY',    'NSE',  'Nifty 50 di NSE (config.yaml saat ini)'),
        ('NIFTY50',  'NSE',  'Nifty 50 nama alternatif'),
        ('NIFTY',    'BSE',  'Nifty 50 di BSE'),
        ('CNXNIFTY', 'NSE',  'CNX Nifty'),
    ],
    'FTSE': [
        ('UKX',      'SP',   'FTSE 100 di Standard & Poors'),
        ('FTSE',     'LSE',  'FTSE 100 di LSE (config.yaml saat ini)'),
        ('UK100',    'TVC',  'FTSE 100 di TVC'),
        ('FTSE100',  'LSE',  'FTSE 100 nama alternatif'),
    ],
}

CONFIRMED = {}

for idx_name, candidates in CANDIDATES.items():
    print(f'\n── Probe {idx_name} ──────────────────────')
    found = False
    for symbol, exchange, desc in candidates:
        try:
            df_test = tv.get_hist(
                symbol=symbol, exchange=exchange,
                interval=Interval.in_15_minute, n_bars=5,
            )
            if df_test is not None and not df_test.empty:
                print(f'  OK     : symbol={symbol!r:12s} exchange={exchange!r:14s} | {desc}')
                if not found:
                    CONFIRMED[idx_name] = {'symbol': symbol, 'exchange': exchange}
                    found = True
            else:
                print(f'  KOSONG : symbol={symbol!r:12s} exchange={exchange!r:14s} | {desc}')
        except Exception as e:
            print(f'  ERROR  : symbol={symbol!r:12s} exchange={exchange!r:14s} | {e}')

print('\n── Ringkasan CONFIRMED ───────────────')
for k, v in CONFIRMED.items():
    print(f'  {k:6s} -> symbol={v["symbol"]!r}, exchange={v["exchange"]!r}')

missing = [i['symbol'] for i in INDICES if i['symbol'] not in CONFIRMED]
if missing:
    print(f'\nWARNING: Tidak ada kandidat valid untuk: {missing}')
    print('  Tambahkan kandidat baru di dict CANDIDATES lalu jalankan ulang.')
else:
    print('\nSemua indeks terkonfirmasi. Lanjut ke cell download.')


## 4. Download 6000 Bar per Indeks

- Menggunakan `CONFIRMED` dari hasil probe
- Retry 3× dengan jeda 10 detik bila koneksi putus
- Cache: kalau file parquet sudah ada dan `FORCE=False`, dilewati


In [ ]:
import time

FORCE = False   # set True untuk re-download meski file sudah ada

def _normalize_df(df, symbol):
    df = df.copy()
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df.columns = [str(c).lower() for c in df.columns]
    df.index = pd.to_datetime(df.index, utc=True)
    ohlcv = ['open', 'high', 'low', 'close', 'volume']
    missing_cols = [c for c in ohlcv if c not in df.columns]
    if missing_cols:
        raise ValueError(
            f'{symbol}: kolom tidak ditemukan: {missing_cols} '
            f'(tersedia: {list(df.columns)})'
        )
    df = df[ohlcv].sort_index()
    df = df[~df.index.duplicated(keep='first')]
    return df


def download_with_retry(tv, symbol, exchange, n_bars, max_retry=3, wait=10):
    for attempt in range(1, max_retry + 1):
        try:
            print(f'  Attempt {attempt}/{max_retry} ...', end=' ')
            df = tv.get_hist(
                symbol=symbol, exchange=exchange,
                interval=Interval.in_15_minute, n_bars=n_bars,
            )
            if df is None or df.empty:
                raise ValueError('get_hist() mengembalikan data kosong.')
            df = _normalize_df(df, symbol)
            print(f'OK ({len(df)} bar)')
            return df
        except Exception as e:
            print(f'GAGAL: {e}')
            if attempt < max_retry:
                print(f'  Menunggu {wait}s ...')
                time.sleep(wait)
    raise RuntimeError(f'Download {symbol} gagal setelah {max_retry} percobaan.')


raw_files = {}
for idx_cfg in INDICES:
    sym = idx_cfg['symbol']
    save_path = paths.raw_parquet(sym)

    if save_path.exists() and not FORCE:
        df_cached = pd.read_parquet(save_path)
        print(f'[{sym}] Cache ditemukan ({len(df_cached)} bar) -> dilewati.')
        raw_files[sym] = df_cached
        continue

    if sym not in CONFIRMED:
        print(f'[{sym}] WARNING: tidak ada kombinasi valid -> dilewati.')
        continue

    conf = CONFIRMED[sym]
    print(f'\n[{sym}] Download {N_BARS} bar '
          f'(symbol={conf["symbol"]!r}, exchange={conf["exchange"]!r}) ...')
    df = download_with_retry(tv, conf['symbol'], conf['exchange'], n_bars=N_BARS)
    raw_files[sym] = df
    save_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(save_path)
    print(f'  Disimpan -> {save_path}')

print('\nDownload selesai.')


## 5. Validasi Hasil

In [ ]:
all_ok = True
rows = []

for idx_cfg in INDICES:
    sym = idx_cfg['symbol']
    save_path = paths.raw_parquet(sym)

    if not save_path.exists():
        print(f'[{sym}] File tidak ditemukan: {save_path}')
        all_ok = False
        continue

    df = pd.read_parquet(save_path)
    n_bars_actual = len(df)
    n_null        = int(df.isnull().sum().sum())
    n_dup         = int(df.index.duplicated().sum())
    date_start    = df.index.min().strftime('%Y-%m-%d')
    date_end      = df.index.max().strftime('%Y-%m-%d')
    is_utc        = str(df.index.tz) == 'UTC'
    cols_ok       = list(df.columns) == ['open', 'high', 'low', 'close', 'volume']
    meets_target  = n_bars_actual >= N_BARS
    meets_min     = n_bars_actual >= N_BARS_MIN

    status = 'OK' if (meets_min and n_null == 0 and n_dup == 0 and is_utc and cols_ok) else 'WARN'
    if status != 'OK':
        all_ok = False

    rows.append({
        'symbol':  sym,
        'bars':    n_bars_actual,
        '>=6000':  'OK' if meets_target else 'FAIL',
        '>=3000':  'OK' if meets_min    else 'FAIL',
        'nulls':   n_null,
        'dups':    n_dup,
        'UTC':     'OK' if is_utc  else 'FAIL',
        'cols':    'OK' if cols_ok else 'FAIL',
        'start':   date_start,
        'end':     date_end,
        'status':  status,
    })

val_df = pd.DataFrame(rows).set_index('symbol')
print(val_df.to_string())
print('\n' + ('Semua validasi lulus.' if all_ok else
              'WARNING: Ada indeks yang tidak memenuhi syarat.'))


## 6. (Opsional) Update config.yaml Bila Exchange Berbeda

Jalankan cell ini kalau probe menemukan exchange berbeda dari `config.yaml`.


In [ ]:
diffs = []
for idx_cfg in INDICES:
    sym = idx_cfg['symbol']
    if sym in CONFIRMED:
        conf_exch = CONFIRMED[sym]['exchange']
        conf_sym  = CONFIRMED[sym]['symbol']
        if conf_exch != idx_cfg['exchange'] or conf_sym != sym:
            diffs.append((sym, idx_cfg['exchange'], conf_sym, conf_exch))

if not diffs:
    print('CONFIRMED cocok dengan config.yaml — tidak ada yang perlu diubah.')
else:
    print('Ditemukan perbedaan:')
    for yaml_sym, yaml_exch, conf_sym, conf_exch in diffs:
        print(f'  {yaml_sym}: exchange {yaml_exch!r} -> {conf_exch!r}')
    answer = input('\nUpdate config.yaml sekarang? [y/N]: ').strip().lower()
    if answer == 'y':
        cfg_path = REPO / 'configs' / 'config.yaml'
        cfg_text = cfg_path.read_text(encoding='utf-8')
        for yaml_sym, yaml_exch, conf_sym, conf_exch in diffs:
            cfg_text = cfg_text.replace(
                f'exchange: {yaml_exch}', f'exchange: {conf_exch}', 1)
        cfg_path.write_text(cfg_text, encoding='utf-8')
        print(f'config.yaml diperbarui -> {cfg_path}')
    else:
        print('Dibatalkan.')


## 7. Upload ke Kaggle sebagai Dataset

In [ ]:
print('File di data/raw/ :')
for f in sorted(paths.raw.glob('*.parquet')):
    size_kb = f.stat().st_size / 1024
    df_tmp  = pd.read_parquet(f)
    print(f'  {f.name:30s}  {size_kb:8.1f} KB  {len(df_tmp)} bar  '
          f'{df_tmp.index[0].date()} -> {df_tmp.index[-1].date()}')
print()
print('Langkah upload ke Kaggle:')
print('  1. Buka https://www.kaggle.com/datasets/new')
print(f'  2. Upload semua .parquet dari: {paths.raw}')
print('  3. Beri nama, misal: hblstm-index-15m-raw')
print('  4. Di Kaggle notebook: Add Data -> cari dataset tersebut')
print('     File tersedia di /kaggle/input/<nama-dataset>/')
print()
print('Atau via Kaggle API:')
print(f'  cd "{paths.raw}"')
print( '  kaggle datasets init -p .')
print( '  # edit dataset-metadata.json')
print( '  kaggle datasets create -p .')


### Snippet untuk `00_run_all.ipynb`

Tambahkan cell ini di `00_run_all.ipynb` **setelah bootstrap**, sebelum `build_dataset()`.  
Ganti `hblstm-index-15m-raw` dengan nama dataset Kaggle Anda.

```python
# Salin data dari Kaggle Dataset ke paths.raw
import shutil
from pathlib import Path

KAGGLE_DATASET_DIR = Path("/kaggle/input/hblstm-index-15m-raw")
if KAGGLE_DATASET_DIR.exists():
    for f in KAGGLE_DATASET_DIR.glob("*.parquet"):
        dest = paths.raw / f.name
        if not dest.exists():
            shutil.copy(f, dest)
            print(f"Copied: {f.name}")
        else:
            print(f"Already exists: {f.name}")
else:
    print(f"Dataset tidak ditemukan di {KAGGLE_DATASET_DIR}")
    print("Pastikan dataset sudah di-Add di sidebar Kaggle.")
```
